# QCar PyramidFuse — Inference Walkthrough

This notebook explains the complete PyramidFusion inference path: LSS, per-agent backbone, aligner, multiscale occupancy-weighted fusion, post-fusion heads, decoding, and NMS.

There is currently **no compatible trained camera-Pyramid checkpoint**. Shape tracing with random Pyramid weights is allowed for education, while the complete production validation path remains ready for the checkpoint produced by the training notebook.

## Phase 0 — Environment and configuration

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import OrderedDict
import os, json, time
from datetime import datetime
import torch
from torch.utils.data import DataLoader

HEAL_ROOT = Path.cwd()
if not (HEAL_ROOT / 'opencood').is_dir(): HEAL_ROOT = (HEAL_ROOT / '..').resolve()
os.chdir(HEAL_ROOT)
import qcar.patches.patch_1cam_loader
import opencood.hypes_yaml.yaml_utils as yaml_utils
from opencood.data_utils.datasets import build_dataset
from opencood.tools import train_utils
from opencood.utils import eval_utils

assert torch.cuda.is_available(), 'HEAL LSS construction requires CUDA in this checkout.'
DEVICE = torch.device('cuda')
USE_AMP = True
CONFIG_PATH = 'qcar/configs/camera_pyramid_onlyfront.yaml'
hypes = yaml_utils.load_yaml(CONFIG_PATH, SimpleNamespace(model_dir=''))
assert hypes.get('test_dir') is None
print('Model:', hypes['model']['core_method'])
print('Validation:', hypes['validate_dir'])

## Phase 1 — Weight policy
Set `MODEL_DIR` to a Pyramid training run to select its single best-development checkpoint, or set `EXACT_CHECKPOINT` to the frozen final-fit checkpoint. Until then, only architecture/shape inspection is permitted.

In [ ]:
MODEL_DIR = None  # Example: HEAL_ROOT/'opencood/logs/qcar_pyramid_YYYYMMDD_HHMMSS'
EXACT_CHECKPOINT = None  # Example: HEAL_ROOT/'opencood/logs/qcar_pyramid_final_.../net_epoch12.pth'
PYRAMID_CHECKPOINT = Path(EXACT_CHECKPOINT) if EXACT_CHECKPOINT is not None else None
if PYRAMID_CHECKPOINT is None and MODEL_DIR is not None:
    candidates = sorted(Path(MODEL_DIR).glob('net_epoch_bestval_at*.pth'))
    assert len(candidates) == 1, candidates
    PYRAMID_CHECKPOINT = candidates[0]
ALLOW_RANDOM_ARCHITECTURE_WALKTHROUGH = True
HAS_TRAINED_WEIGHTS = PYRAMID_CHECKPOINT is not None
if HAS_TRAINED_WEIGHTS:
    PYRAMID_CHECKPOINT = Path(PYRAMID_CHECKPOINT)
    assert PYRAMID_CHECKPOINT.is_file(), PYRAMID_CHECKPOINT
elif not ALLOW_RANDOM_ARCHITECTURE_WALKTHROUGH:
    raise RuntimeError('Provide a Pyramid checkpoint or explicitly allow a random shape walkthrough.')
else:
    print('WARNING: random Pyramid weights; detections and metrics will be disabled.')

## Phase 2 — Deterministic data and camera geometry

In [ ]:
dataset = build_dataset(hypes, visualize=False, train=False)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0,
                    collate_fn=dataset.collate_batch_test)
sample = dataset[0]['ego']['input_m2']
print('Validation frames:', len(dataset))
print('Agents/cameras/image:', tuple(sample['imgs'].shape))
print('Intrinsics:', sample['intrins'])
print('640-to-512 post scale:', float(sample['post_rots'][0, 0, 0, 0]))

## Phase 3 — Instantiate fusion and post-fusion modules
PyramidFusion owns its multiscale backbone, per-level occupancy heads, affine weighted fusion, and multiscale decoder. Detection heads remain outside it and operate after fusion.

In [ ]:
model = train_utils.create_model(hypes)
if HAS_TRAINED_WEIGHTS:
    state = torch.load(PYRAMID_CHECKPOINT, map_location='cpu')
    state = state.get('model_state_dict', state)
    model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()

encoder = model.encoder_m2
agent_backbone = model.backbone_m2
aligner = model.aligner_m2
pyramid_fusion = model.pyramid_backbone
post_fusion_shrinker = model.shrink_conv
post_fusion_heads = OrderedDict([
    ('classification', model.cls_head),
    ('regression', model.reg_head),
    ('direction', model.dir_head),
])
print('Fusion instance:', pyramid_fusion)
print('Pyramid levels:', pyramid_fusion.num_levels)
print('Occupancy heads:', [getattr(pyramid_fusion, f'single_head_{i}')
                            for i in range(pyramid_fusion.num_levels)])
print('Post-fusion shrinker:', post_fusion_shrinker)
print('Post-fusion heads:', post_fusion_heads)

## Phase 4 — Trace the complete fusion path
For every pyramid level: produce per-agent features and occupancy logits → warp both into ego coordinates → softmax occupancy across agents → weighted sum → multiscale decode.

In [ ]:
def shape_of(value):
    if torch.is_tensor(value): return tuple(value.shape)
    if isinstance(value, dict): return {k: shape_of(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [shape_of(v) for v in value]
    return type(value).__name__

trace = {}; handles = []
for name, module in [('LSS encoder', encoder), ('agent backbone', agent_backbone),
                     ('aligner', aligner),
                     ('post-fusion shrinker', post_fusion_shrinker),
                     *[(f'post-fusion {name}', module) for name, module in post_fusion_heads.items()]]:
    handles.append(module.register_forward_hook(
        lambda module, inputs, output, name=name: trace.update({name: shape_of(output)})))
for level in range(pyramid_fusion.num_levels):
    head = getattr(pyramid_fusion, f'single_head_{level}')
    handles.append(head.register_forward_hook(
        lambda module, inputs, output, level=level: trace.update({f'pyramid occupancy level {level}': shape_of(output)})))
handles.append(post_fusion_shrinker.register_forward_pre_hook(
    lambda module, inputs: trace.update({'PyramidFusion decoded output': shape_of(inputs[0])})))
batch = next(iter(loader)); batch = train_utils.to_device(batch, DEVICE)
with torch.inference_mode(), torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
for handle in handles: handle.remove()
print('record_len:', batch['ego']['record_len'].tolist())
print('pairwise transforms:', tuple(batch['ego']['pairwise_t_matrix'].shape))
for stage, shape in trace.items(): print(stage, '->', shape)
print('Occupancy maps:', [tuple(x.shape) for x in output['occ_single_list']])

## Phase 5 — Post-fusion decode and NMS
This block refuses to report random-weight boxes as detections. Supply a trained Pyramid checkpoint first.

In [ ]:
if not HAS_TRAINED_WEIGHTS:
    print('SKIPPED: decode/NMS results would be meaningless with random Pyramid weights.')
else:
    pred_boxes, pred_scores, gt_boxes = dataset.post_process(batch, {'ego': output})
    print('Predictions after NMS:', 0 if pred_boxes is None else len(pred_boxes))
    print('Ground truth:', 0 if gt_boxes is None else len(gt_boxes))
    if pred_scores is not None: print('Top score:', float(pred_scores.max()))
    if pred_boxes is not None: print('Finite decoded boxes:', bool(torch.isfinite(pred_boxes).all()))

## Phase 6 — Complete development-validation inference
When trained weights are loaded, this runs every development-validation frame, reports AP/recall at several IoU thresholds, measures model latency, and writes an auditable JSON report.

In [ ]:
RUN_FULL_VALIDATION = True
IOU_THRESHOLDS = (0.2, 0.3, 0.5, 0.7)
if RUN_FULL_VALIDATION and HAS_TRAINED_WEIGHTS:
    stats = {threshold: {'tp': [], 'fp': [], 'score': [], 'gt': 0}
             for threshold in IOU_THRESHOLDS}
    predicted_frames = 0
    inference_seconds = []
    with torch.inference_mode():
        for item in loader:
            item = train_utils.to_device(item, DEVICE)
            torch.cuda.synchronize(); started = time.perf_counter()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                item_output = model(item['ego'])
            torch.cuda.synchronize(); inference_seconds.append(time.perf_counter()-started)
            boxes, scores, ground_truth = dataset.post_process(item, {'ego': item_output})
            predicted_frames += int(boxes is not None and len(boxes) > 0)
            for threshold in IOU_THRESHOLDS:
                eval_utils.caluclate_tp_fp(boxes, scores, ground_truth, stats, threshold)
    metrics = {}
    for threshold in IOU_THRESHOLDS:
        stat = stats[threshold]
        ap, _, _ = eval_utils.calculate_ap(stats, threshold)
        metrics[str(threshold)] = {'ap': ap, 'gt': stat['gt'],
                                   'tp': sum(stat['tp']), 'fp': sum(stat['fp']),
                                   'recall': sum(stat['tp'])/stat['gt'] if stat['gt'] else None}
    report = {'scope': 'development validation, not final test',
              'checkpoint': str(PYRAMID_CHECKPOINT), 'frames': len(dataset),
              'prediction_frames': predicted_frames, 'amp': USE_AMP,
              'mean_model_seconds': sum(inference_seconds)/len(inference_seconds),
              'metrics': metrics}
    output_dir = HEAL_ROOT/'opencood/logs/qcar_inference_reports'
    output_dir.mkdir(parents=True, exist_ok=True)
    report_path = output_dir/('pyramidfuse_validate_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.json')
    with open(report_path, 'w') as stream: json.dump(report, stream, indent=2)
    print(json.dumps(report, indent=2)); print('Written:', report_path)
elif not HAS_TRAINED_WEIGHTS:
    print('SKIPPED: train PyramidFuse and load its checkpoint before reporting metrics.')
else:
    print('Full validation disabled.')

## Phase 7 — Test policy
`test_dir` remains intentionally null. Freeze the chosen checkpoint and thresholds, then collect a new independent QCar trajectory for the final unbiased evaluation.